In [2]:
%pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [kagglehub]/7 [kagglehub]rmalizer]
Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.9 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 12.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 6.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 2.0 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 23.0 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 911.5/911.5 kB 843.6 kB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 23.5 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [ultralytics] [ultralytics]thop]orm]
Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import random
import shutil
import yaml
import numpy as np
import torch
import kagglehub
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [7]:

SEED = 42
IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 10
MODEL_WEIGHTS = "yolov9c.pt"
OUTPUT_DIR = Path("veylorax_dataset").resolve()
RUNS_DIR = Path("runs/veylorax").resolve()

In [8]:

NAMES = [
    "ambulance", "army vehicle", "auto rickshaw", "bicycle", "bus", "car",
    "garbagevan", "human hauler", "minibus", "minivan", "motorbike", "pickup",
    "policecar", "rickshaw", "scooter", "suv", "taxi",
    "three wheelers -CNG-", "truck", "van", "wheelbarrow"
]

# 1. Télécharger le dataset
root = Path(kagglehub.dataset_download("ashfakyeafi/road-vehicle-images-dataset")) / "trafic_data"
images_dir = root / "train" / "images"
labels_dir = root / "train" / "labels"



# 2. Sélectionner les images et labels disponibles
image_paths = sorted(
    p for p in images_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and (labels_dir / f"{p.stem}.txt").is_file()
)


100%|██████████| 115M/115M [00:02<00:00, 43.5MB/s] 

Extracting files...


In [9]:

# 3. Dataset PyTorch conservé pour inspection / usage personnalisé
class VehicleDataset(Dataset):
    def __init__(self, paths, img_size=IMG_SIZE):
        self.paths = paths
        self.img_size = img_size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image_path = self.paths[idx]
        label_path = labels_dir / f"{image_path.stem}.txt"
        image = Image.open(image_path).convert("RGB").resize((self.img_size, self.img_size))
        image = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
        boxes = []
        for line in label_path.read_text(encoding="utf-8").splitlines():
            values = line.split()
            if len(values) == 5:
                boxes.append([float(v) for v in values])
        targets = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 5)
        return image, targets



# 4. Split reproductible 80/10/10
indices = list(range(len(image_paths)))
random.Random(SEED).shuffle(indices)
n = len(indices)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
train_indices = indices[:n_train]
val_indices = indices[n_train:n_train + n_val]
test_indices = indices[n_train + n_val:]
print(f"Total={n} | Train={len(train_indices)} | Val={len(val_indices)} | Test={len(test_indices)}")


Total=2704 | Train=2163 | Val=270 | Test=271


In [11]:


# 5. DataLoaders PyTorch (non utilisés directement par Ultralytics)
dataset = VehicleDataset(image_paths)
def collate_fn(batch):
    return torch.stack([x[0] for x in batch]), [x[1] for x in batch]
train_loader = DataLoader(Subset(dataset, train_indices), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(Subset(dataset, val_indices), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(Subset(dataset, test_indices), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)



In [12]:


# 6. Matérialiser le même split sur disque pour Ultralytics
# Attention : ce dossier est régénéré à chaque exécution.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
for split in ("train", "val", "test"):
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def copy_split(split_indices, split):
    for idx in split_indices:
        src_image = image_paths[idx]
        src_label = labels_dir / f"{src_image.stem}.txt"
        shutil.copy2(src_image, OUTPUT_DIR / "images" / split / src_image.name)
        shutil.copy2(src_label, OUTPUT_DIR / "labels" / split / src_label.name)

copy_split(train_indices, "train")
copy_split(val_indices, "val")
copy_split(test_indices, "test")

In [13]:
# 7. Écrire data.yaml avec chemins absolus corrects
config = {
    "path": str(OUTPUT_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(NAMES),
    "names": NAMES,
}
yaml_path = OUTPUT_DIR / "data.yaml"
with yaml_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)
print("YAML:", yaml_path)


YAML: /veylorax_dataset/data.yaml


In [14]:
# Important : vérifier que l'ordre NAMES correspond aux IDs des labels originaux.
# 8. Fine-tuning YOLOv9-C pré-entraîné
device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL_WEIGHTS)


In [15]:
results = model.train(
    data=str(yaml_path), epochs=EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, device=device, project=str(RUNS_DIR), name="finetune", exist_ok=True
)


# 9. Utiliser le meilleur checkpoint plutôt que les poids du dernier epoch
best_path = Path(model.trainer.best)
best_model = YOLO(str(best_path))


Ultralytics 8.4.193 🚀 Python-3.12.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A6000, 48550MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/veylorax_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov9c.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=finetune, nbs=64, nms=None, opset=None, optimize=False,

In [16]:
# 10. Évaluer Validation et Test (mAP50, mAP50-95)
for split in ("val", "test"):
    metrics = best_model.val(data=str(yaml_path), split=split, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=device)
    print(f"{split.upper()} | mAP@50={metrics.box.map50:.4f} | mAP@50-95={metrics.box.map:.4f}")

Ultralytics 8.4.193 🚀 Python-3.12.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA RTX A6000, 48550MiB)
YOLOv9c summary (fused): 156 layers, 25,335,439 parameters, 0 gradients, 102.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 926.9±201.3 MB/s, size: 29.2 KB)
val: Scanning /veylorax_dataset/labels/val.cache... 270 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 270/270 125.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 34/34 7.3it/s 4.7s0.3ss
                   all        270       2102      0.529       0.42      0.442      0.296
             ambulance          4          4          0          0     0.0248     0.0137
          army vehicle          1          1      0.577          1      0.995      0.995
         auto rickshaw         15         46      0.671        0.5      0.516      0.322
               bicycle         38         59      0.507      0.305      0.285      0.115
                   bus        

In [17]:
# 11. Prédire une image du Test
sample_image = image_paths[test_indices[0]]
predictions = best_model.predict(source=str(sample_image), imgsz=IMG_SIZE, conf=0.25, save=True, device=device)
print("Image test :", sample_image)
print("Détections :", len(predictions[0].boxes))


image 1/1 /root/.cache/kagglehub/datasets/ashfakyeafi/road-vehicle-images-dataset/versions/2/trafic_data/train/images/Navid_496_jpg.rf.c8fe17ce281b0ca42a0804de70c4b5b8.jpg: 640x384 1 motorbike, 1 suv, 1 van, 18.5ms
Speed: 0.7ms preprocess, 18.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)
Results saved to /runs/detect/predict
Image test : /root/.cache/kagglehub/datasets/ashfakyeafi/road-vehicle-images-dataset/versions/2/trafic_data/train/images/Navid_496_jpg.rf.c8fe17ce281b0ca42a0804de70c4b5b8.jpg
Détections : 3


In [ ]:
# 12. Sauvegarder le meilleur modèle fine-tuné en .pt
saved_path = Path("vehicle_detection.pt").resolve()
shutil.copy2(best_path, saved_path)
print("Modèle sauvegardé :", saved_path)

Modèle sauvegardé : /veylorax.pt
